# 🏎️ McQueen StyleTTS2 — Continue Training (200+ Epochs)
This notebook continues fine-tuning from your existing checkpoint.
Run all cells top to bottom. Training will resume from epoch 50 and push to 250.

In [ ]:
# ─── CELL 1: Mount Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ─── CELL 2: Check GPU ───────────────────────────────────────────────────────
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

In [ ]:
# ─── CELL 3: Install dependencies ───────────────────────────────────────────
!pip install -q SoundFile phonemizer munch einops transformers tqdm librosa matplotlib
!pip install -q torch==2.1.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!apt-get install -q espeak-ng

In [ ]:
# ─── CELL 4: Clone StyleTTS2 ─────────────────────────────────────────────────
import os
if not os.path.exists('/content/StyleTTS2'):
    !git clone https://github.com/yl4579/StyleTTS2 /content/StyleTTS2
os.chdir('/content/StyleTTS2')
!pip install -q -r requirements.txt

In [ ]:
# ─── CELL 5: Copy checkpoint + dataset from Drive ────────────────────────────
import shutil, os

# ‼️ EDIT THESE PATHS if your Drive folder structure is different
DRIVE_CHECKPOINT_ZIP = '/content/drive/MyDrive/mcqueen_trained.zip'   # your existing zip
DRIVE_AUDIO_DIR = '/content/drive/MyDrive/mcqueen_dataset/'           # folder with WAV files (optional, if you want to add more)

# Unzip previous checkpoint
os.makedirs('/content/StyleTTS2/Models/McQueen', exist_ok=True)
!unzip -o "{DRIVE_CHECKPOINT_ZIP}" -d /content/StyleTTS2/Models/McQueen/
print('Checkpoint files:')
!ls -lh /content/StyleTTS2/Models/McQueen/

In [ ]:
# ─── CELL 6: Find latest checkpoint epoch ────────────────────────────────────
import glob, re

ckpts = glob.glob('/content/StyleTTS2/Models/McQueen/epoch_2nd_*.pth')
if ckpts:
    latest = sorted(ckpts, key=lambda x: int(re.search(r'epoch_2nd_(\d+)', x).group(1)))[-1]
    epoch_num = int(re.search(r'epoch_2nd_(\d+)', latest).group(1))
    print(f'Latest checkpoint: {latest} (epoch {epoch_num})')
else:
    # Check for first-stage checkpoints
    ckpts = glob.glob('/content/StyleTTS2/Models/McQueen/epoch_*.pth')
    latest = sorted(ckpts)[-1] if ckpts else None
    print(f'Found checkpoint: {latest}')

In [ ]:
# ─── CELL 7: Verify config.yml exists ────────────────────────────────────────
config_path = '/content/StyleTTS2/Models/McQueen/config.yml'
if not os.path.exists(config_path):
    # Create a config with higher epochs
    print('WARNING: config.yml not found. You need to upload your config.yml from training to Drive.')
    print('Your config should be in the mcqueen_trained.zip. Check the extracted files above.')
else:
    # Patch max_epochs to 250 
    with open(config_path, 'r') as f:
        config_text = f.read()
    
    import re
    # Update epochs to 250
    config_text = re.sub(r'max_epoch:\s*\d+', 'max_epoch: 250', config_text)
    # Ensure save_freq is reasonable
    config_text = re.sub(r'save_freq:\s*\d+', 'save_freq: 10', config_text)

    with open(config_path, 'w') as f:
        f.write(config_text)
    
    print('Config updated: max_epoch=250, save_freq=10')
    !cat "{config_path}" | grep -E 'max_epoch|save_freq|batch_size'

In [ ]:
# ─── CELL 8: Download pre-trained models (ASR, JDC, PLBERT) if missing ────────
import os

def download_if_missing(path, url):
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        !wget -q -O "{path}" "{url}"
        print(f'Downloaded {path}')
    else:
        print(f'Already exists: {path}')

# ASR model
download_if_missing(
    '/content/StyleTTS2/Utils/ASR/epoch_00080.pth',
    'https://huggingface.co/yl4579/StyleTTS2-LibriTTS/resolve/main/Models/LibriTTS/epoch_00080.pth'
)
# JDC F0 model
download_if_missing(
    '/content/StyleTTS2/Utils/JDC/bst.t7',
    'https://github.com/nickoala/jdc/raw/master/bst.t7'
)
print('Pre-trained utilities ready!')

In [ ]:
# ─── CELL 9: Install NLTK data ───────────────────────────────────────────────
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
# ─── CELL 10: RESUME TRAINING (200→250 epochs) ───────────────────────────────
# This will auto-detect and resume from your latest checkpoint.
# Training will run until epoch 250 saving every 10 epochs.
import os
os.chdir('/content/StyleTTS2')

!python train_second.py -p Models/McQueen/config.yml

In [ ]:
# ─── CELL 11: Package & download best checkpoint ─────────────────────────────
import glob, re, shutil, torch, os

ckpts = glob.glob('/content/StyleTTS2/Models/McQueen/epoch_2nd_*.pth')
latest = sorted(ckpts, key=lambda x: int(re.search(r'epoch_2nd_(\d+)', x).group(1)))[-1]
epoch_num = int(re.search(r'epoch_2nd_(\d+)', latest).group(1))
print(f'Best checkpoint: epoch {epoch_num}')

# Prune optimizer states to shrink from ~1.4GB to ~750MB
full = torch.load(latest, map_location='cpu')
pruned = {}
keep_keys = {'net'}
net = full.get('net', full)
# Only keep generator/decoder networks, not discriminators
gen_keys = ['bert', 'bert_encoder', 'predictor', 'text_encoder', 'decoder', 'diffusion', 'style_encoder', 'text_aligner', 'pitch_extractor']
pruned_net = {k: v for k, v in net.items() if any(g in k for g in gen_keys)}
torch.save({'net': pruned_net}, f'/content/mcqueen_model_ep{epoch_num}_pruned.pth')

# Copy config too
shutil.copy('/content/StyleTTS2/Models/McQueen/config.yml', '/content/config.yml')

# Zip together
!zip -j /content/mcqueen_v2_ep{epoch_num}.zip /content/mcqueen_model_ep{epoch_num}_pruned.pth /content/config.yml
print(f'Package ready: /content/mcqueen_v2_ep{epoch_num}.zip')
!ls -lh /content/mcqueen_v2_ep{epoch_num}.zip

In [ ]:
# ─── CELL 12: Download zip to your computer ───────────────────────────────────
from google.colab import files
import glob
zips = glob.glob('/content/mcqueen_v2_*.zip')
files.download(zips[-1])